### 1. Finetune Model : phoBert

In [ ]:
import torch
from transformers import (
    AutoModelForSequenceClassification, 
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    BitsAndBytesConfig
)
from peft import (
    LoraConfig, 
    get_peft_model, 
    prepare_model_for_kbit_training,
    TaskType
)
from datasets import Dataset
from pyvi import ViTokenizer
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

### 2. Training Process

In [ ]:
# -----------------------------
# Device / backend preferences
# -----------------------------
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

#### Data Preparation

In [10]:
import json
def readCorpus(filePath : str):
    with open(filePath, "r", encoding = "utf-8") as f:
        return f.read_json(filePath)

In [ ]:
# Load và preprocessing data
df = pd.read_csv("vietnamese_sentiment.csv")
# Columns: 'text', 'label' (0: tiêu cực, 1: trung tính, 2: tích cực)

# Tách từ tiếng Việt
df['text'] = df['text'].apply(lambda x: ViTokenizer.tokenize(x))

# Split dataset
train_df, temp_df = train_test_split(df, test_size=0.3, stratify=df['label'], random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df['label'], random_state=42)

# Convert to Dataset
train_dataset = Dataset.from_pandas(train_df[['text', 'label']])
val_dataset = Dataset.from_pandas(val_df[['text', 'label']])
test_dataset = Dataset.from_pandas(test_df[['text', 'label']])


#### Setup Model với QLoRA + Quantization

In [ ]:
def setup_model_and_tokenizer(model_name="vinai/phobert-base"):
    """Setup model với 4-bit QLoRA và tối ưu hóa."""
    
    # 1. Cấu hình 4-bit quantization
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,                      # Load weights ở 4-bit
        bnb_4bit_quant_type="nf4",              # NormalFloat 4-bit
        bnb_4bit_compute_dtype=torch.bfloat16,  # Compute ở bfloat16
        bnb_4bit_use_double_quant=True,         # Double quantization
    )
    
    # 2. Load tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    
    # 3. Load model với quantization
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=3,                           # 3 classes sentiment
        quantization_config=bnb_config,
        device_map="auto",                      # Auto distribute
        torch_dtype=torch.bfloat16,
        trust_remote_code=True,
    )
    
    # 4. Gradient checkpointing để tiết kiệm memory
    model.gradient_checkpointing_enable(
        gradient_checkpointing_kwargs={"use_reentrant": False}
    )
    
    # 5. Chuẩn bị model cho k-bit training
    model = prepare_model_for_kbit_training(
        model, 
        use_gradient_checkpointing=True
    )
    
    # 6. Enable input gradients
    try:
        model.enable_input_require_grads()
    except:
        def make_inputs_require_grad(module, input, output):
            output.requires_grad_(True)
        model.get_input_embeddings().register_forward_hook(make_inputs_require_grad)
    
    return model, tokenizer

# Initialize
model, tokenizer = setup_model_and_tokenizer()


#### Config LoRA

In [ ]:
# LoRA configuration
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,         # Sequence Classification
    r=16,                                # Rank (8, 16, 32)
    lora_alpha=32,                       # Scaling factor (thường = 2*r)
    lora_dropout=0.1,                    # Dropout
    target_modules=[                     # Target attention layers
        "query",
        "key", 
        "value",
        "dense"                          # Thêm dense cho performance tốt hơn
    ],
    bias="none",                         # Không train bias
    inference_mode=False,
)

# Apply LoRA
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# Output: trainable params: 1,228,800 / 135,000,000 = 0.91%


#### Tokenization

In [ ]:
def tokenize_function(examples):
    """Tokenize text với padding và truncation."""
    return tokenizer(
        examples['text'],
        padding='max_length',
        truncation=True,
        max_length=256,
        return_tensors=None  # None để tương thích với datasets
    )

# Tokenize datasets
train_dataset = train_dataset.map(tokenize_function, batched=True, remove_columns=['text'])
val_dataset = val_dataset.map(tokenize_function, batched=True, remove_columns=['text'])
test_dataset = test_dataset.map(tokenize_function, batched=True, remove_columns=['text'])

# Rename label column
train_dataset = train_dataset.rename_column("label", "labels")
val_dataset = val_dataset.rename_column("label", "labels")
test_dataset = test_dataset.rename_column("label", "labels")

# Set format
train_dataset.set_format("torch")
val_dataset.set_format("torch")
test_dataset.set_format("torch")


#### Training Arguments với Optimization

In [ ]:
training_args = TrainingArguments(
    output_dir="./phobert-qlora-sentiment",
    
    # Training config
    num_train_epochs=5,
    per_device_train_batch_size=16,        # Tăng được nhờ QLoRA
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=2,         # Effective batch = 16*2 = 32
    
    # Optimization
    learning_rate=3e-4,                    # LoRA dùng LR cao hơn
    weight_decay=0.01,
    warmup_ratio=0.1,                      # 10% warmup steps
    lr_scheduler_type="cosine",            # Cosine decay
    
    # Mixed precision
    bf16=True,                             # BFloat16 training
    
    # Evaluation & Logging
    evaluation_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    save_total_limit=2,                    # Chỉ giữ 2 best checkpoints
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    
    # Memory optimization
    gradient_checkpointing=True,
    optim="paged_adamw_32bit",            # Paged optimizer cho QLoRA
    
    # Other
    report_to="none",                      # Không dùng wandb/tensorboard
    remove_unused_columns=False,
)


#### Metrics

In [ ]:
def compute_metrics(eval_pred):
    """Tính accuracy và F1 score."""
    predictions, labels = eval_pred
    predictions = predictions.argmax(axis=-1)
    
    accuracy = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average='macro')
    
    return {
        'accuracy': accuracy,
        'f1': f1,
    }

# Khởi tạo Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)


#### Training

In [ ]:
# Start training
print("Starting training...")
train_result = trainer.train()

# Print training results
print(f"\nTraining completed!")
print(f"Training loss: {train_result.training_loss:.4f}")

# Evaluate on validation set
eval_results = trainer.evaluate()
print(f"\nValidation Results:")
print(f"Accuracy: {eval_results['eval_accuracy']:.4f}")
print(f"F1 Score: {eval_results['eval_f1']:.4f}")

# Save model
trainer.save_model("./phobert-qlora-final")
tokenizer.save_pretrained("./phobert-qlora-final")


#### Inference

In [ ]:
def predict_sentiment(text, model, tokenizer):
    """Inference function."""
    # Tách từ
    segmented_text = ViTokenizer.tokenize(text)
    
    # Tokenize
    inputs = tokenizer(
        segmented_text,
        padding=True,
        truncation=True,
        max_length=256,
        return_tensors="pt"
    )
    
    # Move to device
    device = model.device
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    # Inference
    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        probs = torch.softmax(logits, dim=-1)
        predicted_class = torch.argmax(probs, dim=-1)
    
    # Map to labels
    labels_map = {0: "Tiêu cực", 1: "Trung tính", 2: "Tích cực"}
    
    return {
        'label': labels_map[predicted_class.item()],
        'confidence': probs[0][predicted_class].item(),
        'probabilities': {
            'Tiêu cực': probs[0][0].item(),
            'Trung tính': probs[0][1].item(),
            'Tích cực': probs[0][2].item(),
        }
    }

# Test inference
test_texts = [
    "Sản phẩm rất tốt, tôi rất hài lòng",
    "Chất lượng tệ, không đáng tiền",
    "Sản phẩm bình thường, không có gì đặc biệt"
]

for text in test_texts:
    result = predict_sentiment(text, model, tokenizer)
    print(f"\nText: {text}")
    print(f"Prediction: {result['label']} (confidence: {result['confidence']:.3f})")
    print(f"Probabilities: {result['probabilities']}")


#### Evaluate on Test sets

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

def evaluate_test_set(model, test_dataset, tokenizer):
    """Evaluate model trên test set."""
    predictions = trainer.predict(test_dataset)
    pred_labels = predictions.predictions.argmax(axis=-1)
    true_labels = predictions.label_ids
    
    # Classification report
    label_names = ["Tiêu cực", "Trung tính", "Tích cực"]
    print("\nClassification Report:")
    print(classification_report(true_labels, pred_labels, target_names=label_names))
    
    # Confusion matrix
    print("\nConfusion Matrix:")
    cm = confusion_matrix(true_labels, pred_labels)
    print(cm)
    
    return pred_labels, true_labels

# Run evaluation
pred_labels, true_labels = evaluate_test_set(model, test_dataset, tokenizer)


# Model 2. Tutorial from Stanford

<a href="https://colab.research.google.com/drive/1Ud5gvvgWn86P70cTHMe8_-6Sqx6LyEAC#scrollTo=X7YRhPGdMPL1" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#### Load Dataset

In [91]:
from datasets import load_dataset, DatasetDict, Dataset
import pandas as pd

#### Load dataset từ huggingface

In [ ]:
dataset = load_dataset("uitnlp/vietnamese_students_feedback", revision="refs/convert/parquet")

In [70]:
dataset

DatasetDict({
    train: Dataset({
        features: ['sentence', 'sentiment', 'topic'],
        num_rows: 11426
    })
    validation: Dataset({
        features: ['sentence', 'sentiment', 'topic'],
        num_rows: 1583
    })
    test: Dataset({
        features: ['sentence', 'sentiment', 'topic'],
        num_rows: 3166
    })
})

#### đọc file

In [87]:
import json
def readJson(filepath):
    with open(filepath, "r", encoding="utf-8") as f:
        return json.load(f)

def read_json(filepath):
    # Pandas có hàm read_json và tự lo việc mở file
    return pd.read_json(filepath)

In [94]:
data = read_json("/Users/theson/Documents/Lab-CS221UIT-NLP/lab4_finetune_model/data/train.json") # dùng pandas đọc thẳng
data1 = readJson("/Users/theson/Documents/Lab-CS221UIT-NLP/lab4_finetune_model/data/train.json")
# Chuyển trực tiếp từ list dict sang Dataset
hf_dataset = Dataset.from_list(data1)
hf_dataset

Dataset({
    features: ['Sentence', 'Emotion'],
    num_rows: 5548
})

In [97]:
hf_dataset['Sentence']

Column(['cho mình xin bài nhạc tên là gì với ạ', 'cho đáng đời con quỷ . về nhà lôi con nhà mày ra mà đánh 😡', 'lo học đi . yêu đương lol gì hay lại thích học sinh học', 'uớc gì sau này về già vẫn có thể như cụ này :))', 'mỗi lần có video của con là cứ coi đi coi lại hoài . cưng con quá .'])

In [65]:
df_train = pd.DataFrame(dataset['train'])
df_train


,sentence,sentiment,topic
0,slide giáo trình đầy đủ .,2,1
1,"nhiệt tình giảng dạy , gần gũi với sinh viên .",2,0
2,đi học đầy đủ full điểm chuyên cần .,0,1
3,chưa áp dụng công nghệ thông tin và các thiết ...,0,0
4,"thầy giảng bài hay , có nhiều bài tập ví dụ ng...",2,0
...,...,...,...
11421,chỉ vì môn game mà em học hai lần mà không qua...,0,1
11422,em cảm ơn cô nhiều .,2,0
11423,giao bài tập quá nhiều .,0,0
11424,"giáo viên dạy dễ hiểu , nhiệt tình .",2,0


#### Thu nhỏ data để test

In [71]:
def truncate(data):
    return {
        'sentence': " ".join(data['sentence'].split()[:50]),
        'sentiment': data['sentiment']
    }

small_dataset = DatasetDict(
    train=dataset['train'].shuffle(seed=1111).select(range(128)).map(truncate),
    val=dataset['train'].shuffle(seed=1111).select(range(128, 160)).map(truncate),
)
small_dataset

Map:   0%|          | 0/128 [00:00<?, ? examples/s]

Map:   0%|          | 0/32 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['sentence', 'sentiment', 'topic'],
        num_rows: 128
    })
    val: Dataset({
        features: ['sentence', 'sentiment', 'topic'],
        num_rows: 32
    })
})

In [77]:
small_dataset['train'][:10]

{'sentence': ['thầy nhiệt tình vui tính , dạy dễ hiểu .',
  'thây dạy rất nhiệt tình và dễ hiểu .',
  'giảng viên nhiệt tình , tận tâm với công việc .',
  'đôi khi em nghĩ thầy cho deadline quá trời nhưng nhờ vậy em học được hơn thầy ạ .',
  'nhưng mà phòng thực hành hiện tại thì chỉ trang bị máy có 1 gb ram thôi .',
  'em cám ơn !',
  'dạy chưa nhiệt tình .',
  'thầy thường 8h mới bắt đầu dạy , lời giảng còn khó hiểu do không cho nhiều ví dụ cụ thể cũng như demo .',
  'vui vẻ , dạy dễ hiểu .',
  'truyền đạt hay tương tác nhiều với sinh viên giúp sinh viên hiểu thêm về môn học .'],
 'sentiment': [2, 2, 2, 0, 0, 1, 0, 0, 2, 2],
 'topic': [0, 0, 0, 0, 2, 3, 0, 0, 0, 0]}

## Task 1: Defining Custom Datasets

There are a few ways to go about defining datasets, but I'm going to show an example using Pytorch Dataloaders. This example uses an encoder-decoder dataaset,the [E2E Dataset](https://arxiv.org/abs/1706.09254), which is maps structured information about restaurants to natural language descriptions.

## Task 2: Pipelines

There are some standard NLP tasks like sentiment classification or question answering where there are already pre-trained (and fine-tuned!) models available through Hugging Face Transformer's [_Pipeline_](https://huggingface.co/docs/transformers/v4.16.2/en/main_classes/pipelines#transformers.pipeline) interface.

For your projects, you likely won't be using it too much, but it's still worth knowing about!

Here's an example with Sentiment Analysis: